# V4 Merged Dataset Audit
## V3 base + Dec-Feb gap rows + BP analytics

Training: Oct 2023 → Feb 2026 | No H2H (leaky) | OOF stacking

**POINTS AUC: 0.637** | **REBOUNDS AUC: 0.593** (honest, no leakage)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)

df = pd.read_csv('nba/features/datasets/xl_merged_training_POINTS_2023_2026.csv')
print(f'Shape: {df.shape}')
print(f'Date range: {df["game_date"].min()} to {df["game_date"].max()}')
print(f'Label rate: {df["label"].mean():.4f}')
print(f'is_home: dtype={df["is_home"].dtype}, home%={df["is_home"].mean()*100:.1f}%')

## Dataset Composition

In [ ]:
# V3 base vs gap rows
df['period'] = df['game_date'].apply(lambda d: 'V3 base (Oct 23 - Dec 25)' if d <= '2025-12-01' else 'Gap (Dec 25 - Feb 26)')
print(df['period'].value_counts())
print()

# Monthly distribution
df['month'] = pd.to_datetime(df['game_date']).dt.to_period('M')
monthly = df['month'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(16, 4))
colors = ['#2ecc71' if str(m) <= '2025-12' else '#e74c3c' for m in monthly.index]
monthly.plot(kind='bar', ax=ax, color=colors)
ax.set_title('Rows per month (green=V3 base, red=gap rows)')
ax.set_ylabel('Props')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout()
plt.show()

## Feature Health Report

In [ ]:
label = df['label']
exclude = {'player_name','game_date','stat_type','opponent_team','source','split','label',
           'game_time','game_id','period','month','actual_points','actual_value','actual_result'}

report = []
for col in sorted(df.columns):
    if col in exclude:
        continue
    s = df[col]
    if s.dtype not in [np.float64, np.int64, float, int, bool]:
        continue
    s = s.astype(float)
    nz = (s != 0).mean() * 100
    nan_pct = s.isna().mean() * 100
    corr = s.corr(label) if s.std() > 0 else 0
    
    if s.std() == 0:
        status = 'DEAD'
    elif nan_pct > 80:
        status = 'MOSTLY NaN'
    elif nz < 10:
        status = 'SPARSE'
    elif nz < 50:
        status = 'WEAK'
    else:
        status = 'OK'
    
    report.append({
        'feature': col, 'nonzero%': round(nz, 1), 'NaN%': round(nan_pct, 1),
        'mean': round(s.mean(), 4), 'std': round(s.std(), 4),
        'corr_label': round(corr, 4), 'abs_corr': round(abs(corr), 4), 'status': status
    })

rdf = pd.DataFrame(report)
print(f'Total features: {len(rdf)}')
for s in ['OK','WEAK','SPARSE','MOSTLY NaN','DEAD']:
    n = (rdf['status'] == s).sum()
    if n > 0:
        print(f'  {s}: {n}')
rdf.sort_values('abs_corr', ascending=False)

## Top 30 Features by |Correlation with Label|

In [ ]:
top = rdf[rdf['status'].isin(['OK','WEAK'])].sort_values('abs_corr', ascending=False).head(30)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#2ecc71' if c > 0 else '#e74c3c' for c in top['corr_label']]
ax.barh(top['feature'][::-1], top['corr_label'][::-1], color=colors[::-1])
ax.set_xlabel('Correlation with label (OVER/UNDER)')
ax.set_title('Top 30 features by |correlation| — V4 merged dataset')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

## BP Analytics Features (new in V4)

In [ ]:
bp_feats = rdf[rdf['feature'].str.contains('bp_|dvp_')].sort_values('abs_corr', ascending=False)
print(f'BP/DVP features: {len(bp_feats)}')
bp_feats[['feature','nonzero%','NaN%','corr_label','status']]

## V3 Base vs Gap Row Quality

In [ ]:
v3_rows = df[df['period'] == 'V3 base (Oct 23 - Dec 25)']
gap_rows = df[df['period'] == 'Gap (Dec 25 - Feb 26)']

# Compare key features between V3 base and gap
key_feats = ['line','ema_points_L3','ema_points_L10','bp_hit_rate_season',
             'bp_projection_diff','team_pace','opponent_pace','is_home',
             'consensus_line','line_spread','num_books_offering']

compare = []
for f in key_feats:
    if f in df.columns:
        v3_mean = v3_rows[f].mean()
        gap_mean = gap_rows[f].mean()
        v3_nan = v3_rows[f].isna().mean() * 100
        gap_nan = gap_rows[f].isna().mean() * 100
        compare.append({'feature': f, 'V3 mean': round(v3_mean, 2), 'Gap mean': round(gap_mean, 2),
                       'V3 NaN%': round(v3_nan, 1), 'Gap NaN%': round(gap_nan, 1)})

pd.DataFrame(compare)

## Gap-Only Features (exist only in gap rows)

In [ ]:
# Features that are all-NaN in V3 base but populated in gap
gap_only = []
for col in df.select_dtypes(include='number').columns:
    if v3_rows[col].isna().all() and not gap_rows[col].isna().all():
        gap_pop = (gap_rows[col].notna() & (gap_rows[col] != 0)).mean() * 100
        gap_only.append({'feature': col, 'gap_populated%': round(gap_pop, 1)})

gap_df = pd.DataFrame(gap_only).sort_values('gap_populated%', ascending=False)
print(f'Gap-only features (NaN for V3, populated for gap): {len(gap_df)}')
gap_df

## Dead & Duplicate Features

In [ ]:
dead = rdf[rdf['status'] == 'DEAD']
print(f'Dead features: {len(dead)}')
if len(dead) > 0:
    print(dead[['feature','mean']].to_string(index=False))

# Highly correlated pairs
numeric = df.select_dtypes(include='number').dropna(axis=1, how='all')
numeric = numeric[[c for c in numeric.columns if numeric[c].std() > 0]]
cm = numeric.corr()
dupes = []
for i, c1 in enumerate(cm.columns):
    for c2 in cm.columns[i+1:]:
        if abs(cm.loc[c1, c2]) > 0.99:
            dupes.append((c1, c2, round(cm.loc[c1, c2], 4)))
print(f'\nHighly correlated pairs (|r| > 0.99): {len(dupes)}')
for a, b, r in sorted(dupes):
    print(f'  {a:40s} <-> {b:40s} r={r}')

## Train/Test Split Distribution

In [ ]:
# 70/30 temporal split (what the trainer uses)
si = int(len(df) * 0.7)
train = df.iloc[:si]
test = df.iloc[si:]

print(f'Train: {len(train)} rows, {train["game_date"].min()} to {train["game_date"].max()}')
print(f'Test:  {len(test)} rows, {test["game_date"].min()} to {test["game_date"].max()}')
print(f'\nLabel rate — Train: {train["label"].mean():.4f}, Test: {test["label"].mean():.4f}')
print(f'Home % — Train: {train["is_home"].mean()*100:.1f}%, Test: {test["is_home"].mean()*100:.1f}%')

# Key feature shift between train and test
shifts = []
for f in ['line','bp_hit_rate_season','ema_points_L3','team_pace','consensus_line','num_books_offering']:
    if f in df.columns:
        tr_m = train[f].mean()
        te_m = test[f].mean()
        shifts.append({'feature': f, 'train_mean': round(tr_m, 3), 'test_mean': round(te_m, 3),
                      'shift': round(te_m - tr_m, 3)})
pd.DataFrame(shifts)

## Key Distributions

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))

plot_feats = ['line', 'bp_hit_rate_season', 'ema_points_L3', 'consensus_line',
              'bp_projection_diff', 'team_pace', 'line_spread', 'num_books_offering']

for i, feat in enumerate(plot_feats):
    ax = axes[i//4][i%4]
    if feat in df.columns:
        over = df[df['label']==1][feat].dropna()
        under = df[df['label']==0][feat].dropna()
        ax.hist(under, bins=40, alpha=0.5, label='UNDER', color='red', density=True)
        ax.hist(over, bins=40, alpha=0.5, label='OVER', color='green', density=True)
        corr = df[feat].corr(df['label'])
        ax.set_title(f'{feat}\ncorr={corr:+.4f}', fontsize=9)
        ax.legend(fontsize=7)
    ax.tick_params(labelsize=7)

plt.suptitle('Feature distributions by OVER/UNDER outcome', fontsize=12)
plt.tight_layout()
plt.show()

## H2H Features (leaky — dropped from training)

In [ ]:
h2h_cols = [c for c in df.columns if c.startswith('h2h_') or c == 'matchup_advantage_score']
if h2h_cols:
    print(f'{len(h2h_cols)} H2H columns still in dataset (dropped during training):')
    for c in sorted(h2h_cols):
        corr = df[c].corr(label) if df[c].std() > 0 else 0
        nz = (df[c] != 0).mean() * 100
        print(f'  {c:40s} corr={corr:+.4f} pop={nz:.1f}%')
    
    # Show why they're leaky
    if 'h2h_avg_points' in df.columns:
        nz = df[df['h2h_avg_points'] > 0].copy()
        nz['h2h_vs_line'] = nz['h2h_avg_points'] - nz['line']
        print(f'\n(h2h_avg - line) corr with label: {nz["h2h_vs_line"].corr(nz["label"]):+.4f}')
        print(f'  OVER mean:  {nz[nz["label"]==1]["h2h_vs_line"].mean():+.2f}')
        print(f'  UNDER mean: {nz[nz["label"]==0]["h2h_vs_line"].mean():+.2f}')
        print('  No separation → H2H from matchup_history table has data leakage (future games included)')
else:
    print('H2H columns already removed from dataset')

## Interactive Inspector

In [ ]:
def inspect(feature_name):
    if feature_name not in df.columns:
        print(f'{feature_name} not found')
        return
    s = df[feature_name].astype(float)
    print(f'Feature: {feature_name}')
    print(f'  dtype: {df[feature_name].dtype}, NaN: {s.isna().sum()} ({s.isna().mean()*100:.1f}%)')
    print(f'  mean: {s.mean():.4f}, std: {s.std():.4f}, min: {s.min():.3f}, max: {s.max():.3f}')
    print(f'  zero: {(s==0).sum()} ({(s==0).mean()*100:.1f}%), corr: {s.corr(label):.4f}')
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].hist(s.dropna(), bins=50, alpha=0.7)
    axes[0].set_title(f'{feature_name} distribution')
    
    axes[1].boxplot([s[label==0].dropna(), s[label==1].dropna()], labels=['UNDER','OVER'])
    axes[1].set_title(f'{feature_name} by outcome')
    
    # V3 base vs gap
    v3_vals = v3_rows[feature_name].dropna() if feature_name in v3_rows.columns else pd.Series()
    gap_vals = gap_rows[feature_name].dropna() if feature_name in gap_rows.columns else pd.Series()
    if len(v3_vals) > 0 and len(gap_vals) > 0:
        axes[2].hist(v3_vals, bins=40, alpha=0.5, label='V3 base', color='green', density=True)
        axes[2].hist(gap_vals, bins=40, alpha=0.5, label='Gap', color='red', density=True)
        axes[2].legend()
        axes[2].set_title('V3 base vs Gap distribution')
    plt.tight_layout()
    plt.show()

# Usage: inspect('bp_hit_rate_season')
inspect('bp_hit_rate_season')

In [ ]:
for f in ['line', 'ema_points_L3', 'bp_projection_diff', 'prop_hit_rate_context',
          'team_pace', 'momentum_short_term', 'days_into_season']:
    inspect(f)